# Fine-tune Mistral-7B-Instruct-v0.3 for the Bangladesh Government Chatbot

This notebook is adapted from the supplied Qwen fine-tuning notebook, but uses **Mistral-7B-Instruct-v0.3** and Mistral's own chat template.

**Training setup**
- Base: `unsloth/mistral-7b-instruct-v0.3-bnb-4bit`
- Method: 4-bit QLoRA with Unsloth
- LoRA rank: 16
- Epochs: 3
- Dataset: `Govt_Chatbot/RAG/unified_train.json` (938 examples in the supplied project)
- Task: Bengali Bangladesh-government-service question answering

> Kaggle: enable a GPU accelerator before running. A T4/P100-class GPU is appropriate for this QLoRA setup.

**Kaggle note:** enable Internet access unless this Hugging Face model is already cached/attached to the notebook.

In [ ]:
# Kaggle environment check
import os, sys, glob, json, zipfile, shutil, random
import numpy as np
import pandas as pd
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())
else:
    print('WARNING: No GPU detected. Enable a Kaggle GPU before training.')

In [ ]:
# Install/upgrade the training stack
!pip install -q -U unsloth unsloth_zoo transformers trl datasets accelerate peft bitsandbytes

In [ ]:
# IMPORTANT: if Kaggle asks for a runtime restart after installation, restart once
# and resume from THIS cell. This cell is self-contained.
import os, sys, glob, json, zipfile, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from unsloth import FastLanguageModel, is_bf16_supported
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

SEED = 3407
MAX_SEQ_LENGTH = 1024
BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/government_mistral_lora"
TRAINER_DIR = "/kaggle/working/mistral-government-checkpoints"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Base model:", BASE_MODEL)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Locate the training data

The supplied project stores the training set at `Govt_Chatbot/RAG/unified_train.json`. The cell below is intentionally flexible: it first looks for that JSON directly in Kaggle inputs, then for the older `government_chat_train.jsonl`, and finally inside any attached ZIP.

In [ ]:
from pathlib import Path


def locate_training_file():
    # 1) Preferred supplied-project file
    candidates = glob.glob('/kaggle/input/**/unified_train.json', recursive=True)
    if candidates:
        return candidates[0]

    # 2) Backward compatibility with the original Qwen notebook
    candidates = glob.glob('/kaggle/input/**/government_chat_train.jsonl', recursive=True)
    if candidates:
        return candidates[0]

    # 3) Look inside attached ZIP files
    for zip_path in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        try:
            with zipfile.ZipFile(zip_path) as zf:
                members = zf.namelist()
                preferred = [m for m in members if m.endswith('Govt_Chatbot/RAG/unified_train.json')]
                fallback = [m for m in members if m.endswith('unified_train.json')]
                matches = preferred or fallback
                if matches:
                    extract_dir = Path('/kaggle/working/mistral_train_data')
                    extract_dir.mkdir(parents=True, exist_ok=True)
                    member = matches[0]
                    zf.extract(member, extract_dir)
                    return str(extract_dir / member)
        except zipfile.BadZipFile:
            pass

    raise FileNotFoundError(
        'Could not find unified_train.json or government_chat_train.jsonl. '
        'Attach the Govt_Chatbot ZIP/dataset to the Kaggle notebook.'
    )

TRAIN_DATA_PATH = locate_training_file()
print('Training file:', TRAIN_DATA_PATH)

In [ ]:
# Load raw training examples
raw_dataset = load_dataset(
    "json",
    data_files=TRAIN_DATA_PATH,
    split="train",
)

print(raw_dataset)
print("Columns:", raw_dataset.column_names)
print("Examples:", len(raw_dataset))
print("\nFirst example:")
print(raw_dataset[0])

required = {"instruction", "input", "output"}
missing = required - set(raw_dataset.column_names)
assert not missing, f"Missing required columns: {missing}"


In [ ]:
# Dataset diagnostics
pdf = raw_dataset.to_pandas()

for col in ["instruction", "input", "output"]:
    pdf[col] = pdf[col].fillna("").astype(str)

print("Examples:", len(pdf))
print("Non-empty input fields:", (pdf["input"].str.strip() != "").sum())
print("Average instruction characters:", round(pdf["instruction"].str.len().mean(), 1))
print("Average answer characters:", round(pdf["output"].str.len().mean(), 1))
print("Maximum answer characters:", int(pdf["output"].str.len().max()))

if "domain" in pdf.columns:
    print("\nDomain counts:")
    print(pdf["domain"].value_counts())


## Load Mistral-7B in 4-bit and attach LoRA

The seven projection modules below match the LoRA setup used in your Qwen experiment and are also the standard Unsloth recommendation for strong QLoRA adaptation.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('EOS token:', repr(tokenizer.eos_token))
print('PAD token:', repr(tokenizer.pad_token))
print('Tokenizer vocab size:', len(tokenizer))

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

model.print_trainable_parameters()

## Format the 938 examples with Mistral's tokenizer

Do **not** use Qwen's `<|im_start|> ... <|im_end|>` tokens here. `tokenizer.apply_chat_template(...)` lets the installed Mistral tokenizer produce its native instruction format.

In [ ]:
SYSTEM_PROMPT = (
    'তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। '
    'প্রশ্নের সরাসরি, সংক্ষিপ্ত এবং নির্ভুল উত্তর বাংলায় দাও। '
    'শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। '
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
)


def formatting(example):
    instruction = str(example.get('instruction', '') or '').strip()
    input_text = str(example.get('input', '') or '').strip()
    answer = str(example.get('output', '') or '').strip()

    user_text = instruction
    if input_text:
        user_text += '\n' + input_text

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': answer},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Most Mistral templates already terminate assistant turns with EOS.
    # Add it only if the rendered template did not do so.
    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token

    return {'text': text}

formatted_dataset = raw_dataset.map(
    formatting,
    remove_columns=raw_dataset.column_names,
)

print(formatted_dataset)

In [ ]:
# Inspect one rendered training sample and its token length
sample_text = formatted_dataset[0]["text"]
print(sample_text)
print("\nToken length:", len(tokenizer(sample_text, add_special_tokens=False)["input_ids"]))


In [ ]:
# Check truncation risk across all 938 examples before training
lengths = []
for row in formatted_dataset:
    lengths.append(len(tokenizer(row['text'], add_special_tokens=False)['input_ids']))

lengths = np.array(lengths)
print('Token length median:', int(np.median(lengths)))
print('Token length p95:', int(np.percentile(lengths, 95)))
print('Token length max:', int(lengths.max()))
print(f'Examples longer than {MAX_SEQ_LENGTH}:', int((lengths > MAX_SEQ_LENGTH).sum()))

if (lengths > MAX_SEQ_LENGTH).any():
    print('WARNING: Increase MAX_SEQ_LENGTH before training if this count is not zero.')

## Train

This keeps the main Qwen experiment hyperparameters so the comparison is meaningful: rank 16, alpha 16, batch 2, accumulation 4, learning rate `2e-4`, and 3 epochs. Unsloth recommends roughly 1–3 epochs for instruction datasets as a good starting range.

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    args=SFTConfig(
        output_dir=TRAINER_DIR,

        # Dataset handling
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        dataset_num_proc=2,

        # Training
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Optimizer
        learning_rate=2e-4,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,

        # Precision
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),

        # Stability / logging
        max_grad_norm=1.0,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=2,
        report_to='none',
        seed=SEED,
    ),
)

print('Trainer ready.')

In [ ]:
# Start the full fine-tuning run
trainer_stats = trainer.train()
trainer_stats

In [ ]:
# Save LoRA adapters + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training history for later comparison
history_path = '/kaggle/working/mistral_training_history.csv'
pd.DataFrame(trainer.state.log_history).to_csv(history_path, index=False)

run_config = {
    'stage': 'Mistral-7B QLoRA fine-tuning',
    'base_model': BASE_MODEL,
    'training_examples': len(raw_dataset),
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0.0,
    'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'seed': SEED,
}
with open('/kaggle/working/mistral_finetune_run_config.json', 'w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print('Saved adapter:', OUTPUT_DIR)
print('Saved history:', history_path)

In [ ]:
# Create a downloadable ZIP of the LoRA adapter
zip_path = shutil.make_archive(
    '/kaggle/working/government_mistral_lora',
    'zip',
    OUTPUT_DIR,
)
print('Created:', zip_path)

## Sanity-check the fine-tuned model

The same tokenizer/chat template is used at inference time. This is essential; using a different template after training can cause gibberish, repetition, or poor stopping behavior.

In [ ]:
FastLanguageModel.for_inference(model)


def generate_answer(question, context=''):
    question = str(question or '').strip()
    context = '' if context is None else str(context).strip()

    if context:
        user_text = (
            f'প্রশ্ন: {question}\n\n'
            'প্রাসঙ্গিক তথ্য:\n'
            f'{context}\n\n'
            'উপরের প্রাসঙ্গিক তথ্য ব্যবহার করে প্রশ্নের সরাসরি ও সংক্ষিপ্ত উত্তর বাংলায় দাও।'
        )
    else:
        user_text = question

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print(generate_answer('NID আবেদন করতে কোথায় যেতে হবে?'))

## Optional: test a RAG-style prompt

Use the exact same `generate_answer(question, context)` function in your Mistral RAG notebook. Start with the **top 3 reranked passages**, not all five, then compare Top-1/Top-3/Top-5 experimentally.

In [ ]:
# Example only — replace the context with text from your retriever.
example_context = (
    'ই-পাসপোর্ট আবেদনের সাধারণ পাঁচটি ধাপ হলো: প্রথমে আবেদনকারীর এলাকায় '
    'ই-পাসপোর্ট সেবা চালু আছে কি না যাচাই করা, এরপর অনলাইনে আবেদন পূরণ করা, '
    'নির্ধারিত ফি পরিশোধ করা, পাসপোর্ট অফিসে গিয়ে বায়োমেট্রিক এনরোলমেন্ট '
    'সম্পন্ন করা এবং শেষে পাসপোর্ট সংগ্রহ করা।'
)

print(generate_answer('ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?', example_context))

## Outputs to keep from Kaggle

After the notebook finishes, save these files from `/kaggle/working/`:

- `government_mistral_lora.zip` — the trained LoRA adapter
- `government_mistral_lora/` — unzipped adapter + tokenizer
- `mistral_training_history.csv` — loss/log history
- `mistral_finetune_run_config.json` — reproducibility settings

You do **not** need to merge the full 7B model just to use it in your RAG pipeline. Load the same Mistral base model together with this adapter.

## Full test-set evaluation

This section evaluates the fine-tuned model on the **entire available test set**.

Important:
- `TEST_LIMIT = None` means **no 100-example cap** is used.
- Predictions are saved incrementally so the evaluation can resume after interruption.
- All final metrics are computed from the complete prediction table.
- The evaluator supports the project-style `instruction` / `input` / `output` fields and several common question/answer aliases.


In [ ]:
# Install evaluation dependencies
!pip install -q -U sacrebleu rouge-score rapidfuzz bert-score


In [ ]:
# Locate and load the FULL test set
import os, glob, json, zipfile, re, math
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import load_dataset

TEST_LIMIT = None   # IMPORTANT: None = evaluate ALL test examples
FULL_TEST_PREDICTIONS = "/kaggle/working/mistral_full_test_predictions.csv"
FULL_TEST_METRICS = "/kaggle/working/mistral_full_test_metrics.csv"
FULL_TEST_METRICS_JSON = "/kaggle/working/mistral_full_test_metrics.json"

def locate_test_file():
    preferred_names = [
        "unified_test.json",
        "government_chat_test.jsonl",
        "test.json",
        "test.jsonl",
    ]

    # 1) Directly attached files
    for name in preferred_names:
        matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if matches:
            return matches[0]

    # 2) Search inside attached ZIP files
    for zip_path in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        try:
            with zipfile.ZipFile(zip_path) as zf:
                members = zf.namelist()
                for name in preferred_names:
                    matches = [m for m in members if m.endswith(name)]
                    if matches:
                        extract_dir = Path("/kaggle/working/mistral_test_data")
                        extract_dir.mkdir(parents=True, exist_ok=True)
                        member = matches[0]
                        zf.extract(member, extract_dir)
                        return str(extract_dir / member)
        except zipfile.BadZipFile:
            pass

    raise FileNotFoundError(
        "Could not find the test set. Attach unified_test.json, "
        "government_chat_test.jsonl, test.json, or test.jsonl."
    )

TEST_DATA_PATH = locate_test_file()
print("Test file:", TEST_DATA_PATH)

test_dataset = load_dataset(
    "json",
    data_files=TEST_DATA_PATH,
    split="train",
)

print("Full test-set size:", len(test_dataset))
print("Columns:", test_dataset.column_names)

if TEST_LIMIT is None:
    eval_dataset = test_dataset
else:
    eval_dataset = test_dataset.select(range(min(TEST_LIMIT, len(test_dataset))))

print("Examples selected for evaluation:", len(eval_dataset))
assert len(eval_dataset) == len(test_dataset), (
    f"Full-set evaluation expected, but only {len(eval_dataset)} / "
    f"{len(test_dataset)} examples were selected."
)


In [ ]:
# Flexible extraction of question/reference fields

def first_nonempty(example, names):
    for name in names:
        if name in example:
            value = example.get(name)
            if value is not None and str(value).strip():
                return str(value).strip()
    return ""

def get_question(example):
    instruction = first_nonempty(
        example,
        ["instruction", "question", "query", "prompt", "user", "input_question"]
    )
    input_text = first_nonempty(example, ["input", "context_input"])

    # For the project's instruction/input/output structure, combine instruction + input
    if "instruction" in example and instruction:
        return instruction + (("\n" + input_text) if input_text else "")

    return instruction or input_text

def get_reference(example):
    return first_nonempty(
        example,
        ["output", "answer", "reference", "response", "target", "gold"]
    )

# Validate the whole test set before expensive generation
missing_q = 0
missing_r = 0
for ex in eval_dataset:
    missing_q += (get_question(ex) == "")
    missing_r += (get_reference(ex) == "")

print("Missing questions:", missing_q)
print("Missing references:", missing_r)

assert missing_q == 0, "Some test rows have no recognizable question field."
assert missing_r == 0, "Some test rows have no recognizable reference-answer field."


In [ ]:
# Generate predictions for the ENTIRE test set.
# Resume-safe: if the CSV already contains earlier predictions, they are reused.

from tqdm.auto import tqdm
import torch

FastLanguageModel.for_inference(model)

existing = pd.DataFrame()
if os.path.exists(FULL_TEST_PREDICTIONS):
    try:
        existing = pd.read_csv(FULL_TEST_PREDICTIONS)
        print("Existing saved predictions:", len(existing))
    except Exception as e:
        print("Could not read old prediction file; starting clean:", e)
        existing = pd.DataFrame()

saved_by_index = {}
if not existing.empty and "test_index" in existing.columns:
    for _, row in existing.iterrows():
        try:
            idx = int(row["test_index"])
            pred = row.get("prediction", "")
            ref = row.get("reference", "")
            q = row.get("question", "")
            if pd.notna(pred) and str(pred).strip():
                saved_by_index[idx] = {
                    "test_index": idx,
                    "question": "" if pd.isna(q) else str(q),
                    "reference": "" if pd.isna(ref) else str(ref),
                    "prediction": str(pred),
                }
        except Exception:
            pass

rows = []

for idx, ex in enumerate(tqdm(eval_dataset, total=len(eval_dataset), desc="Full test evaluation")):
    if idx in saved_by_index:
        rows.append(saved_by_index[idx])
        continue

    question = get_question(ex)
    reference = get_reference(ex)

    prediction = generate_answer(question)

    row = {
        "test_index": idx,
        "question": question,
        "reference": reference,
        "prediction": prediction,
    }
    rows.append(row)

    # Save frequently so interruption does not destroy completed work
    if (idx + 1) % 10 == 0 or (idx + 1) == len(eval_dataset):
        temp_df = pd.DataFrame(rows).sort_values("test_index")
        temp_df.to_csv(FULL_TEST_PREDICTIONS, index=False, encoding="utf-8-sig")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pred_df = pd.DataFrame(rows).sort_values("test_index").reset_index(drop=True)
pred_df.to_csv(FULL_TEST_PREDICTIONS, index=False, encoding="utf-8-sig")

print("\nCompleted predictions:", len(pred_df))
print("Expected full test size:", len(eval_dataset))
assert len(pred_df) == len(eval_dataset), "Prediction count does not match full test-set size."
assert pred_df["test_index"].nunique() == len(eval_dataset), "Duplicate/missing test indices found."

display(pred_df.head())


In [ ]:
# Compute ALL evaluation metrics from the COMPLETE test set

import re
import unicodedata
from collections import Counter
from rapidfuzz.fuzz import ratio as fuzzy_ratio
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bertscore

assert len(pred_df) == len(eval_dataset), (
    "Metrics must only be computed after predictions exist for the full test set."
)

predictions = pred_df["prediction"].fillna("").astype(str).tolist()
references = pred_df["reference"].fillna("").astype(str).tolist()

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    # Remove punctuation/symbols while preserving letters/numbers from all scripts
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] not in {"P", "S"})
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_f1(pred, ref):
    p = normalize_text(pred).split()
    r = normalize_text(ref).split()

    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0

    common = Counter(p) & Counter(r)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(r)
    return 2 * precision * recall / (precision + recall)

# Exact Match
em_scores = [
    float(normalize_text(p) == normalize_text(r))
    for p, r in zip(predictions, references)
]

# Fuzzy similarity (0-1)
fuzzy_scores = [
    fuzzy_ratio(normalize_text(p), normalize_text(r)) / 100.0
    for p, r in zip(predictions, references)
]

# Token F1
token_f1_scores = [
    token_f1(p, r)
    for p, r in zip(predictions, references)
]

# Sentence BLEU averaged across all test examples
bleu_scores = [
    sacrebleu.sentence_bleu(p, [r]).score / 100.0
    for p, r in zip(predictions, references)
]

# ROUGE-1, ROUGE-2, ROUGE-L
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)
rouge_results = [rouge.score(r, p) for p, r in zip(predictions, references)]
rouge1_scores = [x["rouge1"].fmeasure for x in rouge_results]
rouge2_scores = [x["rouge2"].fmeasure for x in rouge_results]
rougel_scores = [x["rougeL"].fmeasure for x in rouge_results]

# BERTScore using a multilingual encoder
# This is computed for ALL prediction/reference pairs.
P, R, F1 = bertscore(
    predictions,
    references,
    model_type="xlm-roberta-large",
    batch_size=8,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)

bert_p = P.cpu().numpy()
bert_r = R.cpu().numpy()
bert_f1 = F1.cpu().numpy()

metrics = {
    "test_examples": int(len(pred_df)),
    "exact_match": float(np.mean(em_scores)),
    "fuzzy_match": float(np.mean(fuzzy_scores)),
    "token_f1": float(np.mean(token_f1_scores)),
    "bleu": float(np.mean(bleu_scores)),
    "rouge1_f1": float(np.mean(rouge1_scores)),
    "rouge2_f1": float(np.mean(rouge2_scores)),
    "rougeL_f1": float(np.mean(rougel_scores)),
    "bertscore_precision": float(np.mean(bert_p)),
    "bertscore_recall": float(np.mean(bert_r)),
    "bertscore_f1": float(np.mean(bert_f1)),
}

# Add per-example scores to the prediction file as well
pred_df["exact_match"] = em_scores
pred_df["fuzzy_match"] = fuzzy_scores
pred_df["token_f1"] = token_f1_scores
pred_df["bleu"] = bleu_scores
pred_df["rouge1_f1"] = rouge1_scores
pred_df["rouge2_f1"] = rouge2_scores
pred_df["rougeL_f1"] = rougel_scores
pred_df["bertscore_precision"] = bert_p
pred_df["bertscore_recall"] = bert_r
pred_df["bertscore_f1"] = bert_f1

pred_df.to_csv(FULL_TEST_PREDICTIONS, index=False, encoding="utf-8-sig")

metrics_df = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in metrics.items()]
)
metrics_df.to_csv(FULL_TEST_METRICS, index=False)

with open(FULL_TEST_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\n===== FULL TEST-SET EVALUATION =====")
print("Evaluated examples:", len(pred_df), "/", len(test_dataset))
for k, v in metrics.items():
    if k == "test_examples":
        print(f"{k:24s}: {int(v)}")
    else:
        print(f"{k:24s}: {v:.6f}")

print("\nSaved:")
print(FULL_TEST_PREDICTIONS)
print(FULL_TEST_METRICS)
print(FULL_TEST_METRICS_JSON)

display(metrics_df)


### Full-test evaluation outputs

After the evaluation completes, keep these files from `/kaggle/working/`:

- `mistral_full_test_predictions.csv` — every test question, reference answer, generated answer, and per-example scores.
- `mistral_full_test_metrics.csv` — aggregate metrics computed across the full test set.
- `mistral_full_test_metrics.json` — the same aggregate metrics in JSON format.

The notebook asserts that the number of evaluated predictions equals the number of examples in the loaded test set, preventing an accidental 100-example evaluation from being reported as a full-test result.
